# LLM-CTIA: Cyber Threat Intelligence Aggregator

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import re
import feedparser
import pandas as pd
from bs4 import BeautifulSoup
import plotly.express as px

# -----------------------------------
# IOC Patterns
# -----------------------------------
patterns = {
    "IP": r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b",
    "Domain": r"\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,6}\b",
    "CVE": r"CVE-\d{4}-\d{4,7}",
    "Hash": r"\b[a-fA-F0-9]{40}\b"
}

# -----------------------------------
# Static MITRE mapping based on CVE (basic placeholder)
# -----------------------------------
mitre_lookup = {
    "CVE-2019-0708": ["T1133", "T1203"],  # BlueKeep
    "CVE-2021-41773": ["T1190"],
    "CVE-2024-XXXX": ["T1068"],
}

# -----------------------------------
# IOC Extraction Function
# -----------------------------------
def extract_iocs(text):
    extracted = {k: re.findall(v, text) for k, v in patterns.items()}
    extracted = {k: list(set(v)) for k, v in extracted.items() if v}

    # Add MITRE Techniques if CVEs are present
    mitre_techniques = []
    for cve in extracted.get("CVE", []):
        mitre_techniques.extend(mitre_lookup.get(cve, []))
    if mitre_techniques:
        extracted["MITRE_Techniques"] = list(set(mitre_techniques))

    return extracted

# -----------------------------------
# 1. CISA Feed Parser
# -----------------------------------
feed_url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.xml"
feed = feedparser.parse(feed_url)

cisa_entries = []
for e in feed.entries[:10]:
    summary = BeautifulSoup(e.get("summary", ""), "html.parser").get_text()
    row = {"Source": "CISA", "Text": summary}
    row.update(extract_iocs(summary))
    cisa_entries.append(row)

df_cisa = pd.DataFrame(cisa_entries)

# -----------------------------------
# 2. Dark Web Feed (manual samples)
# -----------------------------------
dark_samples = [
    "Selling 0-day exploit for Apache Struts, payload drops Cobalt Strike beacon. CVE-2024-XXXX",
    "Access to 10,000 RDP servers with admin creds. Includes healthcare orgs.",
    "New ALPHV ransomware variant bypasses EDR, using DLL sideloading.",
    "Suspicious domain found: evil-malware.com distributing ransomware.",
    "Hash found in malware sample: b19dbf7e9243a1a2e2d9c98c4328e1ad48c7e9dc"
]

df_dark = pd.DataFrame([
    {"Source": "DarkWeb", "Text": sample, **extract_iocs(sample)}
    for sample in dark_samples
])

# -----------------------------------
# 3. Shodan-style Data
# -----------------------------------
shodan_data = [
    {
        "ip": "192.0.2.10",
        "port": 3389,
        "service": "RDP",
        "vuln": "BlueKeep (CVE-2019-0708)"
    },
    {
        "ip": "203.0.113.5",
        "port": 80,
        "service": "Apache",
        "vuln": "CVE-2021-41773"
    }
]

df_shodan = pd.DataFrame([
    {
        "Source": "Shodan",
        "Text": f"Device at {entry['ip']} running {entry['service']} on port {entry['port']}, vulnerable to {entry['vuln']}.",
        **extract_iocs(f"{entry['ip']} {entry['vuln']}")
    }
    for entry in shodan_data
])

# -----------------------------------
# Combine All
# -----------------------------------
df_all = pd.concat([df_cisa, df_dark, df_shodan], ignore_index=True)
print(df_all[["Source", "Text", "IP", "Domain", "CVE", "Hash", "MITRE_Techniques"]])

# -----------------------------------
# Visualization
# -----------------------------------
source_counts = df_all["Source"].value_counts().reset_index()
source_counts.columns = ["Source", "Count"]

fig = px.bar(source_counts, x="Source", y="Count", title="Threat Intel Sources Processed")
fig.show()
